# ScamRoad: Universal Cloud Recovery & Diagnostics Notebook
Inspects cloud storage, validates checkpoint integrity, recovers stale locks, and resumes training on whichever cloud platform is currently active.


### Cell 1: Detect Platform & Profile Hardware


In [ ]:
from environment_manager import EnvironmentManager
env_mgr = EnvironmentManager()
print(f'[ENVIRONMENT] Active Platform : {env_mgr.platform_name.upper()}')
print(f'[HARDWARE] GPU Model       : {env_mgr.hardware.get("gpu_name", "CPU")}')
print(f'[HARDWARE] VRAM (GB)        : {env_mgr.hardware.get("vram_gb", 0.0)}')
print(f'[HARDWARE] System RAM (GB)  : {env_mgr.hardware.get("ram_total_gb", 0.0)}')


### Cell 2: Connect Authorized Persistent Cloud Storage


In [ ]:
from pathlib import Path
if env_mgr.platform_name == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    cloud_root = Path('/content/drive/MyDrive/scam_road')
elif env_mgr.platform_name == 'kaggle':
    cloud_root = Path('/kaggle/working/scam_road')
else:
    cloud_root = Path('./data/cloud_storage_simulation').resolve()
cloud_root.mkdir(parents=True, exist_ok=True)
print(f'[STORAGE] Connected to: {cloud_root}')


### Cell 3: Scan Interrupted Experiments & Checkpoints


In [ ]:
import json
configs_dir = cloud_root / 'experiment_configs'
configs_dir.mkdir(parents=True, exist_ok=True)
print('=' * 65)
print('          SCAMROAD CLOUD EXPERIMENT REGISTRY SCAN             ')
print('=' * 65)
experiments = list(configs_dir.iterdir())
if not experiments:
    print('No previous experiments found. Initializing fresh registry.')
for exp_path in experiments:
    state_file = exp_path / 'experiment_state.json'
    if state_file.exists():
        state = json.loads(state_file.read_text(encoding='utf-8'))
        print(f'• Experiment : {state.get("experiment_id")}')
        print(f'  Status     : {state.get("status")}')
        print(f'  Platform   : {state.get("current_platform")}')
        print(f'  Last Epoch : {state.get("current_epoch")} / {state.get("target_epochs")}')
print('=' * 65)


### Cell 4: Validate Checkpoint Integrity & Recover Stale Locks


In [ ]:
from checkpoint_manager import CheckpointManager
from experiment_manager import ExperimentManager
exp_id = 'scamroad_yolo11n_multicloud'
ckpt_mgr = CheckpointManager(cloud_root, exp_id)
exp_mgr = ExperimentManager(cloud_root, exp_id)

# Check lock status
locked, lock_data = exp_mgr.is_locked()
if locked:
    print(f'[LOCK STATUS] Active lock held by {lock_data.get("platform")}. Checking lease...')
else:
    print('[LOCK STATUS] Lock is free or stale lease recovered.')

# Verify latest checkpoint
latest_dir, manifest = ckpt_mgr.load_latest_checkpoint()
if latest_dir and manifest:
    v = ckpt_mgr.verify_checkpoint(latest_dir)
    print(f'[CHECKPOINT VERIFICATION] Status: {v["valid"]} | Epoch: {v["epoch"]} | Step: {v["global_step"]}')
else:
    print('[CHECKPOINT VERIFICATION] No valid prior checkpoint found. Fresh initialization required.')


### Cell 5: Resume Training from Interrupted State


In [ ]:
from cloud_orchestrator import CloudOrchestrator
orchestrator = CloudOrchestrator(config_path='cloud_config.yaml')
session = orchestrator.prepare_session(experiment_id=exp_id)
print(f'[RECOVERY PLAN] Action: {session["action"]} | Resume Epoch: {session.get("start_epoch", 0)}')

recovery_results = orchestrator.execute_training(
    experiment_id=exp_id,
    model_name='yolo11n.pt',
    target_epochs=50,
    batch_size=16,
)
print('[RECOVERY EXECUTION RESULT]:', recovery_results)
